# Inside Steam — Current Data Enrichment

## 1. Web Scraping: Steam Top Sellers

### Objective

The main Steam dataset represents a historical snapshot from
March 2025.

This notebook collects a smaller sample of current Steam data
using HTML web scraping, BeautifulSoup and API requests.

The objective is to demonstrate external data collection,
enrich the historical dataset and explore current market signals.

The sample focuses on Steam's Top Sellers page. It is a
popularity-based sample and is not representative of the
entire Steam catalogue.

Collection dates and data sources will be documented
to distinguish current observations from historical data.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

from datetime import datetime, timezone
from pathlib import Path

In [2]:
# Steam Top Sellers page

url = "https://store.steampowered.com/search/"

params = {
    "filter": "topsellers",
    "category1": "998",
    "l": "english",
    "cc": "us"
}

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=20
)

response.raise_for_status()

print("Status code:", response.status_code)
print("HTML length:", len(response.content))

Status code: 200
HTML length: 710824


In [3]:
soup = BeautifulSoup(
    response.content,
    "html.parser"
)

print("Page title:", soup.title.get_text(strip=True))

game_cards = soup.select("a.search_result_row")

print("Number of cards found:", len(game_cards))

Page title: Steam Search
Number of cards found: 50


In [4]:
for card in game_cards[:5]:

    title = card.select_one("span.title")

    print(
        "APPID:", card.get("data-ds-appid"),
        "| NAME:", title.get_text(strip=True) if title else None
    )

APPID: 230410 | NAME: Warframe
APPID: 3669870 | NAME: CONTROL Resonant
APPID: 1867240 | NAME: WARDOGS
APPID: 730 | NAME: Counter-Strike 2
APPID: 1142710 | NAME: Total War: WARHAMMER III


In [5]:
games_data = []

collection_time = datetime.now(
    timezone.utc
).isoformat(timespec="seconds")

for position, card in enumerate(game_cards, start=1):

    appid = card.get("data-ds-appid")
    title = card.select_one("span.title")
    product_url = card.get("href")

    # Keep only games with a valid Steam ID and title
    if not appid or not appid.isdigit() or not title:
        continue

    games_data.append({
        "appid": int(appid),
        "name": title.get_text(strip=True),
        "page_position": position,
        "product_url": product_url,
        "collected_at_utc": collection_time
    })

df_current = pd.DataFrame(games_data)

df_current.head()

,appid,name,page_position,product_url,collected_at_utc
0,230410,Warframe,1,https://store.steampowered.com/app/230410/Warf...,2026-09-24T13:53:38+00:00
1,3669870,CONTROL Resonant,2,https://store.steampowered.com/app/3669870/CON...,2026-09-24T13:53:38+00:00
2,1867240,WARDOGS,3,https://store.steampowered.com/app/1867240/WAR...,2026-09-24T13:53:38+00:00
3,730,Counter-Strike 2,4,https://store.steampowered.com/app/730/Counter...,2026-09-24T13:53:38+00:00
4,1142710,Total War: WARHAMMER III,5,https://store.steampowered.com/app/1142710/Tot...,2026-09-24T13:53:38+00:00


In [6]:
print("Games collected:", len(df_current))

print(
    "Duplicate APPIDs:",
    df_current["appid"].duplicated().sum()
)

print("\nMissing values:")
print(df_current.isna().sum())

print("\nFirst five games:")
display(df_current.head())

Games collected: 50
Duplicate APPIDs: 0

Missing values:
appid               0
name                0
page_position       0
product_url         0
collected_at_utc    0
dtype: int64

First five games:


,appid,name,page_position,product_url,collected_at_utc
0,230410,Warframe,1,https://store.steampowered.com/app/230410/Warf...,2026-09-24T13:53:38+00:00
1,3669870,CONTROL Resonant,2,https://store.steampowered.com/app/3669870/CON...,2026-09-24T13:53:38+00:00
2,1867240,WARDOGS,3,https://store.steampowered.com/app/1867240/WAR...,2026-09-24T13:53:38+00:00
3,730,Counter-Strike 2,4,https://store.steampowered.com/app/730/Counter...,2026-09-24T13:53:38+00:00
4,1142710,Total War: WARHAMMER III,5,https://store.steampowered.com/app/1142710/Tot...,2026-09-24T13:53:38+00:00


In [7]:
output_path = Path("../data/raw")
output_path.mkdir(parents=True, exist_ok=True)

df_current.to_csv(
    output_path / "steam_top_sellers_2026.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [8]:
# Test the API with our first collected game

appid = int(df_current.iloc[0]["appid"])

api_url = "https://store.steampowered.com/api/appdetails"

params = {
    "appids": appid,
    "cc": "us",
    "l": "english"
}

api_response = requests.get(
    api_url,
    params=params,
    headers=headers,
    timeout=20
)

api_response.raise_for_status()

# Convert the JSON response into a Python dictionary
api_data = api_response.json()

print("Status code:", api_response.status_code)
print("API response type:", type(api_data))
print("Success:", api_data[str(appid)]["success"])

Status code: 200
API response type: <class 'dict'>
Success: True


In [9]:
game_info = api_data[str(appid)]["data"]

print("Name:", game_info.get("name"))
print("Type:", game_info.get("type"))
print("Free:", game_info.get("is_free"))
print("Release:", game_info.get("release_date"))

print("\nAvailable fields:")
print(list(game_info.keys()))

Name: Warframe
Type: game
Free: True
Release: {'coming_soon': False, 'date': 'Mar 25, 2013'}

Available fields:
['type', 'name', 'steam_appid', 'required_age', 'is_free', 'controller_support', 'dlc', 'detailed_description', 'about_the_game', 'short_description', 'supported_languages', 'header_image', 'capsule_image', 'capsule_imagev5', 'website', 'pc_requirements', 'mac_requirements', 'linux_requirements', 'legal_notice', 'ext_user_account_notice', 'developers', 'publishers', 'packages', 'package_groups', 'platforms', 'metacritic', 'categories', 'genres', 'screenshots', 'movies', 'recommendations', 'achievements', 'release_date', 'support_info', 'background', 'background_raw', 'content_descriptors', 'ratings']


In [10]:
# Extract selected fields from the API response

price_info = game_info.get("price_overview")

# Free games have a price of zero.
# Missing prices for paid games remain unknown.
if game_info.get("is_free"):
    current_price = 0.0
elif price_info:
    current_price = price_info.get("final", 0) / 100
else:
    current_price = None


# Extract genre names from the nested JSON structure
genres = [
    genre["description"]
    for genre in game_info.get("genres", [])
]


# Create one structured record
game_record = {
    "appid": appid,
    "name": game_info.get("name"),
    "type": game_info.get("type"),
    "is_free": game_info.get("is_free"),
    "current_price_usd": current_price,
    "release_date": game_info.get("release_date", {}).get("date"),
    "coming_soon": game_info.get("release_date", {}).get("coming_soon"),
    "genres": genres,
    "recommendations": game_info.get("recommendations", {}).get("total"),
    "short_description": game_info.get("short_description")
}

game_record

{'appid': 230410,
 'name': 'Warframe',
 'type': 'game',
 'is_free': True,
 'current_price_usd': 0.0,
 'release_date': 'Mar 25, 2013',
 'coming_soon': False,
 'genres': ['Action', 'RPG', 'Free To Play'],
 'recommendations': 2881,
 'short_description': 'Awaken as an unstoppable warrior and battle alongside your friends in this story-driven free-to-play online action game'}

## 2. Automated API Data Collection

After validating the API response for a single game, the extraction
process is automated using a reusable Python function.

The function requests game details using Steam App IDs, extracts
selected variables from the JSON response and handles unavailable
data.

A small batch is tested before collecting the full sample.

In [11]:
import time

def fetch_game_details(appid):

    params = {
        "appids": appid,
        "cc": "us",
        "l": "english"
    }

    response = requests.get(
        api_url,
        params=params,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    api_data = response.json()
    result = api_data.get(str(appid), {})

    # Some products may not have available API details
    if not result.get("success"):
        return None

    info = result["data"]

    # Extract pricing information
    price_info = info.get("price_overview")

    if info.get("is_free"):
        current_price = 0.0
    elif price_info and price_info.get("currency") == "USD":
        current_price = price_info["final"] / 100
    else:
        current_price = None

    # Extract nested fields
    release = info.get("release_date") or {}

    genres = [
        genre["description"]
        for genre in info.get("genres", [])
    ]

    recommendations = info.get("recommendations") or {}

    return {
        "appid": appid,
        "name": info.get("name"),
        "type": info.get("type"),
        "is_free": info.get("is_free"),
        "current_price_usd": current_price,
        "release_date": release.get("date"),
        "coming_soon": release.get("coming_soon"),
        "genres": genres,
        "recommendations": recommendations.get("total"),
        "short_description": info.get("short_description")
    }

In [12]:
# Keep our previously validated Warframe record
test_records = [game_record]

# Select three additional Steam App IDs
test_appids = df_current["appid"].iloc[1:4].tolist()

for appid in test_appids:

    try:
        record = fetch_game_details(appid)

        if record is not None:
            test_records.append(record)
            print(f"Collected: {record['name']}")
        else:
            print(f"No API data available for: {appid}")

    except requests.RequestException as error:
        print(f"Request failed for {appid}: {error}")
        break

    # Avoid sending requests too quickly
    time.sleep(2)

Collected: CONTROL Resonant
Collected: WARDOGS
Collected: Counter-Strike 2


In [13]:
df_api_test = pd.DataFrame(test_records)

display(
    df_api_test[
        [
            "appid",
            "name",
            "type",
            "current_price_usd",
            "coming_soon",
            "genres",
            "recommendations"
        ]
    ]
)

,appid,name,type,current_price_usd,coming_soon,genres,recommendations
0,230410,Warframe,game,0.00,False,"[Action, RPG, Free To Play]",2881.0
1,3669870,CONTROL Resonant,game,59.99,True,"[Action, Adventure, RPG]",NaN
2,1867240,WARDOGS,game,39.99,False,"[Action, Indie, Massively Multiplayer, Simulat...",74721.0
3,730,Counter-Strike 2,game,0.00,False,"[Action, Free To Play]",5281968.0


In [14]:
# Start with the four games already collected
all_records = list(test_records)

# Keep track of collected App IDs to avoid duplicates
collected_appids = {
    record["appid"]
    for record in all_records
}

# Collect the remaining games
for appid in df_current["appid"]:

    if appid in collected_appids:
        continue

    try:
        record = fetch_game_details(int(appid))

        if record is not None:
            all_records.append(record)
            collected_appids.add(appid)

            print(f"{len(all_records)} - Collected: {record['name']}")

        else:
            print(f"No API data available: {appid}")

    except (requests.RequestException, ValueError) as error:
        print(f"Collection stopped at {appid}: {error}")
        break

    time.sleep(2)

5 - Collected: Total War: WARHAMMER III
6 - Collected: SILENT HILL: Townfall
7 - Collected: Apex Legends™
8 - Collected: Disney Dreamlight Valley
9 - Collected: Marvel Rivals
10 - Collected: HELLDIVERS™ 2
11 - Collected: Graveyard Keeper 2
12 - Collected: Aniimo
13 - Collected: ACE COMBAT 8: WINGS OF THEVE
14 - Collected: EA SPORTS FC™ 27
15 - Collected: Dressmaker
16 - Collected: Black Desert
17 - Collected: Borderlands 4
18 - Collected: RuneScape: Dragonwilds
19 - Collected: Destiny 2
20 - Collected: KINGDOM HEARTS -HD 1.5+2.5 ReMIX-
21 - Collected: Limbus Company
22 - Collected: Diablo® IV
23 - Collected: Tom Clancy's Rainbow Six Siege
24 - Collected: AION 2
25 - Collected: Overwatch®
26 - Collected: Once Human
27 - Collected: Dune: Awakening
28 - Collected: NBA 2K27
29 - Collected: War Thunder
30 - Collected: Dead by Daylight
31 - Collected: FINAL FANTASY XIV Online
32 - Collected: EA SPORTS™ Madden NFL 27
33 - Collected: REMNANT II®
34 - Collected: Happy Wheels
35 - Collected: Ste

In [15]:
import json

df_api = pd.DataFrame(all_records)

# JSON is suitable for preserving nested structures such as genre lists.
api_output = output_path / "steam_api_details_2026.json"

with open(api_output, "w", encoding="utf-8") as file:
    json.dump(all_records, file, ensure_ascii=False, indent=2)

print("API records collected:", len(df_api))
print("Unique APPIDs:", df_api["appid"].nunique())
print("Data saved successfully!")

API records collected: 50
Unique APPIDs: 50
Data saved successfully!


In [16]:
print("Shape:", df_api.shape)

print("\nProduct types:")
print(df_api["type"].value_counts(dropna=False))

print("\nUpcoming games:")
print(df_api["coming_soon"].value_counts(dropna=False))

print("\nMissing values:")
print(df_api.isna().sum())

Shape: (50, 10)

Product types:
type
game    50
Name: count, dtype: int64

Upcoming games:
coming_soon
False    44
True      6
Name: count, dtype: int64

Missing values:
appid                 0
name                  0
type                  0
is_free               0
current_price_usd     0
release_date          0
coming_soon           0
genres                0
recommendations      13
short_description     0
dtype: int64


## 3. Data Cleaning and Integration

The HTML scraping and API collection produced two complementary
datasets containing 50 Steam games.

Both datasets are merged using the unique Steam App ID.

Basic quality checks are performed to verify data completeness,
identifier consistency, product types and missing values.

Upcoming games are retained but clearly identified to avoid
inappropriate comparisons with released games.

In [17]:
# Rename the API name to distinguish it from the HTML name
df_api_merge = df_api.rename(
    columns={"name": "api_name"}
)

# Merge HTML and API datasets
df_enriched = df_current.merge(
    df_api_merge,
    on="appid",
    how="left",
    validate="one_to_one"
)

print("Final dataset shape:", df_enriched.shape)

display(df_enriched.head())

Final dataset shape: (50, 14)


,appid,name,page_position,product_url,collected_at_utc,api_name,type,is_free,current_price_usd,release_date,coming_soon,genres,recommendations,short_description
0,230410,Warframe,1,https://store.steampowered.com/app/230410/Warf...,2026-09-24T13:53:38+00:00,Warframe,game,True,0.00,"Mar 25, 2013",False,"[Action, RPG, Free To Play]",2881.0,Awaken as an unstoppable warrior and battle al...
1,3669870,CONTROL Resonant,2,https://store.steampowered.com/app/3669870/CON...,2026-09-24T13:53:38+00:00,CONTROL Resonant,game,False,59.99,"Sep 24, 2026",True,"[Action, Adventure, RPG]",NaN,Explore a warped Manhattan on the brink of par...
2,1867240,WARDOGS,3,https://store.steampowered.com/app/1867240/WAR...,2026-09-24T13:53:38+00:00,WARDOGS,game,False,39.99,"Sep 10, 2026",False,"[Action, Indie, Massively Multiplayer, Simulat...",74721.0,WARDOGS is a TACTICAL ALL OUT WARFARE FPS wher...
3,730,Counter-Strike 2,4,https://store.steampowered.com/app/730/Counter...,2026-09-24T13:53:38+00:00,Counter-Strike 2,game,True,0.00,"Aug 21, 2012",False,"[Action, Free To Play]",5281968.0,"For over two decades, Counter-Strike has offer..."
4,1142710,Total War: WARHAMMER III,5,https://store.steampowered.com/app/1142710/Tot...,2026-09-24T13:53:38+00:00,Total War: WARHAMMER III,game,False,59.99,"Feb 16, 2022",False,"[Action, Strategy]",105050.0,The cataclysmic conclusion to the Total War: W...


In [18]:
print("Total games:", len(df_enriched))

print(
    "Duplicate APPIDs:",
    df_enriched["appid"].duplicated().sum()
)

print(
    "Missing API records:",
    df_enriched["api_name"].isna().sum()
)

print(
    "Upcoming games:",
    df_enriched["coming_soon"].sum()
)

print(
    "Missing recommendations:",
    df_enriched["recommendations"].isna().sum()
)

print(
    "Games without genres:",
    df_enriched["genres"].apply(
        lambda genres: len(genres) == 0
        if isinstance(genres, list)
        else True
    ).sum()
)

print(
    "Different names between HTML and API:",
    (
        df_enriched["name"] != df_enriched["api_name"]
    ).sum()
)

Total games: 50
Duplicate APPIDs: 0
Missing API records: 0
Upcoming games: 6
Missing recommendations: 13
Games without genres: 0
Different names between HTML and API: 0


## 4. Historical Dataset Comparison

The enriched sample is compared with the historical March 2025
Steam dataset using Steam App IDs.

The objective is to identify which currently collected games
were already present in the historical dataset and which
are absent from that snapshot.

This comparison helps evaluate the overlap between the
two datasets before considering any temporal analysis.

Absence from the historical dataset does not necessarily
mean that a game was released after March 2025.

In [3]:
# Define the correct project paths
project_root = Path.cwd().parent

raw_path = project_root / "data/raw"

historical_path = (
    project_root / "data/processed/games_cleaned_march2025.csv"
)

# Restore the previously collected HTML data
df_current = pd.read_csv(
    raw_path / "steam_top_sellers_2026.csv"
)

# Restore the previously collected API data
with open(
    raw_path / "steam_api_details_2026.json",
    "r",
    encoding="utf-8"
) as file:
    all_records = json.load(file)

df_api = pd.DataFrame(all_records)

# Reconstruct the enriched dataset
df_enriched = df_current.merge(
    df_api.rename(columns={"name": "api_name"}),
    on="appid",
    how="left",
    validate="one_to_one"
)

# Load historical Steam App IDs
df_historical_ids = pd.read_csv(
    historical_path,
    usecols=["appid"]
)

# Final verification
print("Current sample:", len(df_enriched))
print("Historical dataset:", len(df_historical_ids))
print("Missing API records:", df_enriched["api_name"].isna().sum())

Current sample: 50
Historical dataset: 94948
Missing API records: 0


In [4]:
# Compare current sample with historical dataset

df_enriched["present_in_march2025"] = (
    df_enriched["appid"].isin(df_historical_ids["appid"])
)

print("Historical dataset overlap:")

print(
    df_enriched["present_in_march2025"].value_counts()
)

print("\nGames absent from the historical dataset:")

display(
    df_enriched.loc[
        ~df_enriched["present_in_march2025"],
        ["appid", "name", "release_date", "coming_soon"]
    ]
)

Historical dataset overlap:
present_in_march2025
True     25
False    25
Name: count, dtype: int64

Games absent from the historical dataset:


,appid,name,release_date,coming_soon
1,3669870,CONTROL Resonant,"Sep 24, 2026",True
2,1867240,WARDOGS,"Sep 10, 2026",False
5,1636440,SILENT HILL: Townfall,"Sep 23, 2026",False
10,4358690,Graveyard Keeper 2,"Sep 22, 2026",False
11,4126040,Aniimo,"Sep 15, 2026",False
12,2288340,ACE COMBAT 8: WINGS OF THEVE,"Oct 1, 2026",True
13,4080220,EA SPORTS FC™ 27,"Sep 24, 2026",True
14,4019220,Dressmaker,"Sep 21, 2026",False
16,1285190,Borderlands 4,"Sep 11, 2025",False
17,1374490,RuneScape: Dragonwilds,"Sep 15, 2026",False


In [5]:
# Convert API release dates to datetime

df_enriched["release_date_parsed"] = pd.to_datetime(
    df_enriched["release_date"],
    format="%b %d, %Y",
    errors="coerce"
)

# Historical dataset cutoff
historical_cutoff = pd.Timestamp("2025-03-10")

# Focus on games absent from the historical dataset
new_games = df_enriched[
    ~df_enriched["present_in_march2025"]
].copy()

print("Games absent from historical dataset:", len(new_games))

print(
    "Released after historical snapshot:",
    (
        (new_games["release_date_parsed"] > historical_cutoff)
        & (new_games["coming_soon"] == False)
    ).sum()
)

print(
    "Upcoming games:",
    new_games["coming_soon"].sum()
)

print(
    "Missing or unparseable release dates:",
    new_games["release_date_parsed"].isna().sum()
)

Games absent from historical dataset: 25
Released after historical snapshot: 19
Upcoming games: 6
Missing or unparseable release dates: 0


In [6]:
# Save the final enriched dataset

processed_path = project_root / "data/processed"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

# Prepare a copy for CSV export
df_export = df_enriched.copy()

# Serialize genre lists as JSON strings
df_export["genres"] = df_export["genres"].apply(
    json.dumps
)

# Export the processed dataset
df_export.to_csv(
    processed_path / "steam_current_enriched_2026.csv",
    index=False
)

print("Final dataset saved successfully!")
print("Shape:", df_export.shape)

Final dataset saved successfully!
Shape: (50, 16)


## 5. Current Sample Analysis

This section explores the composition and commercial
characteristics of the 50 games collected in September 2026.

The analysis focuses on:
- Historical versus recently released games
- Pricing and free-to-play distribution
- Genre representation

The sample is drawn from Steam's Top Sellers page
and is not representative of the entire Steam market.

In [7]:
# Classify games by historical presence and release status

df_sample = df_enriched.copy()

df_sample["sample_segment"] = "Historical game"

df_sample.loc[
    ~df_sample["present_in_march2025"],
    "sample_segment"
] = "Recently released"

df_sample.loc[
    df_sample["coming_soon"],
    "sample_segment"
] = "Upcoming"

segment_summary = (
    df_sample["sample_segment"]
    .value_counts()
    .rename_axis("segment")
    .reset_index(name="games")
)

segment_summary["share_pct"] = (
    segment_summary["games"] / len(df_sample) * 100
).round(2)

display(segment_summary)

,segment,games,share_pct
0,Historical game,25,50.0
1,Recently released,19,38.0
2,Upcoming,6,12.0


In [8]:
# Focus on released games

released_games = df_sample[
    df_sample["coming_soon"] == False
].copy()

# Summary by historical segment

pricing_summary = (
    released_games
    .groupby("sample_segment")
    .agg(
        games=("appid", "count"),
        free_games=("is_free", "sum"),
        average_price_usd=("current_price_usd", "mean")
    )
)

# Calculate median price among paid games only

paid_games = released_games[
    released_games["is_free"] == False
]

paid_medians = (
    paid_games
    .groupby("sample_segment")["current_price_usd"]
    .median()
)

pricing_summary["median_paid_price_usd"] = paid_medians

pricing_summary["free_share_pct"] = (
    pricing_summary["free_games"]
    / pricing_summary["games"]
    * 100
)

display(pricing_summary.round(2))

,games,free_games,average_price_usd,median_paid_price_usd,free_share_pct
sample_segment,,,,,
Historical game,25,13,11.50,19.99,52.00
Recently released,19,3,29.12,29.99,15.79


In [9]:
# Explore genre representation

genre_summary = (
    released_games[["appid", "genres"]]
    .explode("genres")
    .dropna(subset=["genres"])
    .groupby("genres")["appid"]
    .nunique()
    .sort_values(ascending=False)
    .head(8)
    .reset_index(name="games")
)

genre_summary["share_pct"] = (
    genre_summary["games"]
    / len(released_games)
    * 100
).round(2)

display(genre_summary)

,genres,games,share_pct
0,Action,32,72.73
1,Adventure,21,47.73
2,RPG,20,45.45
3,Free To Play,16,36.36
4,Simulation,13,29.55
5,Massively Multiplayer,12,27.27
6,Casual,11,25.00
7,Strategy,8,18.18


### Current Sample Analysis — Key Findings

The analysis of the September 2026 sample produced three
main observations.

**1. Sample composition**

The collected sample contains 50 Steam games:
- 25 games already present in the March 2025 dataset.
- 19 games released after the historical snapshot.
- 6 upcoming games.

This enrichment adds more recent titles and textual
information that can support further analysis.

**2. Pricing differences**

Among released games, the historical group contains
a substantially higher proportion of free games (52%)
than the recently released group (15.79%).

The median price among paid games is $19.99 for the
historical group and $29.99 for recently released games.

These differences describe the collected sample only
and should not be generalized to the broader Steam market.

**3. Genre representation**

Action is the most represented genre among the
44 released games, appearing in 72.73% of them.

Adventure (47.73%) and RPG (45.45%) are also highly
represented.

This genre distribution differs from the historical
catalogue, illustrating the importance of considering
sampling methods when comparing datasets.

### Business Relevance

The current sample provides a complementary view of
Steam's market at a more recent collection date.

The collected descriptions offer additional textual
information for AI-assisted feature engineering.

Games present in both the historical and current
datasets may also serve as examples when interpreting
the future market segmentation results.

### Limitations

The sample is small and originates from Steam's
Top Sellers search page.

It is not representative of the complete market,
and differences between historical and recently
released games should not be interpreted as general
market trends or causal relationships.

In [10]:
# Save the enriched dataset with sample classification

df_final = df_sample.copy()

# Serialize genre lists for CSV export
df_final["genres"] = df_final["genres"].apply(
    json.dumps
)

final_path = (
    project_root
    / "data/processed/steam_current_enriched_2026.csv"
)

df_final.to_csv(
    final_path,
    index=False
)

print("Final enriched dataset updated!")
print("Shape:", df_final.shape)

Final enriched dataset updated!
Shape: (50, 17)


## 5. Data Enrichment Summary

### Data Collection

A current sample of 50 Steam games was collected on
September 24, 2026.

Two complementary collection methods were used:

- **HTML web scraping:** BeautifulSoup was used to extract
  game names, Steam App IDs, product URLs and page positions
  from Steam's Top Sellers search page.

- **API and JSON:** Steam Store API requests were used to
  retrieve additional information, including prices,
  release dates, genres, recommendations and game descriptions.

### Data Quality

The HTML and API datasets were merged using Steam App IDs.

The final dataset contains 50 unique games with no missing
API records or duplicate identifiers.

All 50 products were identified as games, including
6 upcoming titles.

Recommendations were unavailable for 13 games.
These missing values were preserved rather than replaced
with zero.

### Historical Dataset Comparison

The current sample was compared with the March 2025
historical dataset containing 94,948 Steam games.

Results:

- 25 games were already present in the historical dataset.
- 25 games were absent from the historical dataset.
- Of these 25 games, 19 were released after the historical
  snapshot and 6 were marked as upcoming.

This demonstrates how current data collection can
complement a historical dataset by introducing more
recently released and upcoming titles.

### Limitations

The sample was collected from Steam's Top Sellers page
and is not representative of the entire Steam catalogue.

Page position should not be interpreted as an exact
sales ranking.

The historical and current datasets represent different
collection dates. Direct comparisons must account for
these temporal differences.

Upcoming games and games with unavailable recommendations
require separate treatment in performance analyses.

### Final Conclusion

This notebook demonstrates an end-to-end data enrichment
workflow combining HTML web scraping, API requests, JSON
processing, data cleaning and dataset integration.

A sample of 50 Steam games was collected and compared
with the historical March 2025 dataset.

The analysis identified differences in pricing, free-to-play
representation and genre distribution between the groups
within our current sample.

These findings are exploratory and cannot be generalized
to the entire Steam market due to the limited and
non-representative sample.

Beyond demonstrating data collection techniques,
this enrichment provides additional information that
can support the next stages of the project.

In particular, the collected game descriptions offer
an opportunity for AI-assisted feature engineering,
while games present in both datasets may help illustrate
the future machine learning segmentation results.